# 02 - Descriptive Analysis

Pre-legalization trends by treatment status. The parallel trends assumption lives or dies here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

## Fatality trends: treated vs never-treated states

In [ ]:
primary_outcome = "total_fatalities_per_100k"

if primary_outcome not in fars.columns:
    # Fallback if per-100k not available
    primary_outcome = "total_fatalities"
    print(f"Warning: using raw counts (no population data). Add Census pop for rates.")

treated_states   = leg[leg['retail_sales_year'].notna()]['state'].tolist()
untreated_states = leg[leg['retail_sales_year'].isna()]['state'].tolist()

avg_by_year = (
    fars
    .assign(group=fars['state'].map(lambda s: 'Treated' if s in treated_states else 'Never Treated'))
    .groupby(['year','group'])[primary_outcome]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11,5))
for grp, color in [('Treated','steelblue'),('Never Treated','orange')]:
    sub = avg_by_year[avg_by_year['group']==grp]
    ax.plot(sub['year'], sub[primary_outcome], marker='o', ms=4, lw=1.8, label=grp, c=color)
ax.set_xlabel("Year")
ax.set_ylabel(primary_outcome.replace('_',' ').title())
ax.set_title("Traffic Fatality Trends: Treated vs Never-Treated States\n(pre-treatment parallel trends visual check)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "02_fatality_trends.png", bbox_inches='tight')
plt.show()

## Individual state event plots — early adopters

In [ ]:
early = leg[leg['retail_sales_year'].isin([2014,2015])]['state'].tolist()

fig, axes = plt.subplots(1, len(early), figsize=(5*len(early), 4), sharey=True)
for ax, state in zip(axes if len(early) > 1 else [axes], early):
    s = fars[fars['state']==state].sort_values('year')
    treat_yr = leg[leg['state']==state]['retail_sales_year'].values[0]
    ax.plot(s['year'], s[primary_outcome], marker='o', ms=4, lw=1.5, c='steelblue')
    ax.axvline(treat_yr, color='red', ls='--', lw=1.2, label=f'Legalization ({int(treat_yr)})')
    ax.set_title(state)
    ax.set_xlabel("Year")
plt.suptitle("Early-Adopter States — Fatalities Around Legalization", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_early_adopter_trends.png", bbox_inches='tight')
plt.show()

## Cohort sizes and treatment timing

In [ ]:
cohort_counts = (
    leg[leg['retail_sales_year'].notna()]
    .groupby('retail_sales_year')
    .agg(n_states=('state','count'), states=('state', lambda x: ', '.join(sorted(x))))
    .reset_index()
    .sort_values('retail_sales_year')
)
print("Treatment cohorts:")
print(cohort_counts.to_string(index=False))